# 02 — Bearing vs. Direction

`compute_bearing` returns a number. That number is the **initial bearing** — the compass direction you face at the moment of departure.

For short distances, the bearing stays roughly constant along the path. For long distances, it does not. The Earth curves underneath you, and what started as a northeast heading gradually shifts as you travel.

This notebook separates two things students often conflate: *bearing* (a number computed at a single point) and *direction* (what the path actually does over its length).

In [ ]:
import math
import json
from pathlib import Path

def compute_bearing(p1, p2):
    """Bearing from p1 to p2. Inputs: [lon, lat]. Output: degrees [0, 360)."""
    lon1, lat1 = math.radians(p1[0]), math.radians(p1[1])
    lon2, lat2 = math.radians(p2[0]), math.radians(p2[1])
    d_lon = lon2 - lon1
    x = math.sin(d_lon) * math.cos(lat2)
    y = math.cos(lat1) * math.sin(lat2) - math.sin(lat1) * math.cos(lat2) * math.cos(d_lon)
    return (math.degrees(math.atan2(x, y)) + 360) % 360

def haversine_km(p1, p2):
    """Great-circle distance in km. Inputs: [lon, lat]."""
    R = 6371.0
    lon1, lat1 = math.radians(p1[0]), math.radians(p1[1])
    lon2, lat2 = math.radians(p2[0]), math.radians(p2[1])
    dlat, dlon = lat2 - lat1, lon2 - lon1
    a = math.sin(dlat/2)**2 + math.cos(lat1) * math.cos(lat2) * math.sin(dlon/2)**2
    return R * 2 * math.asin(math.sqrt(a))

with open(Path("data/bearing_examples.json")) as f:
    examples = json.load(f)

print("Ready.")

## 1. Bearing Is an Initial Direction, Not a Constant

When you call `compute_bearing(p1, p2)`, you get one number. That number describes the direction to face **at p1** to head toward p2. It says nothing about what happens to your heading as you travel.

On a flat surface, the direction stays constant — walk northeast and you keep walking northeast forever. On a sphere, that is not true.

As you travel along a great circle (the shortest path between two points on a sphere), the lines of longitude you cross are converging. You are not moving in a straight line through space — you are arcing along a curved surface. Your compass heading **changes continuously** even if you never turn.

For short distances (under ~200 km), this effect is small enough to ignore. For long routes — city to city, continental distances, missile trajectories — the drift is significant and must be accounted for.

## 2. Measuring Bearing Drift Along a Path

The clearest way to see bearing drift is to walk a path in small steps and compute the bearing at each step. If the heading were constant, every bearing would be identical. On a sphere, they shift.

The function below interpolates `n` equally-spaced intermediate points between p1 and p2 along a straight line in coordinate space (not a true great circle, but close enough for visualization), then computes the bearing from each point to the next.

In [ ]:
def bearing_along_path(p1, p2, n=10):
    """
    Interpolate n+1 points along the line from p1 to p2,
    then compute the bearing from each point to the next.
    Returns a list of (fraction_traveled, bearing) tuples.
    """
    results = []
    for i in range(n):
        t0 = i / n
        t1 = (i + 1) / n
        a = [p1[0] + t0 * (p2[0] - p1[0]), p1[1] + t0 * (p2[1] - p1[1])]
        b = [p1[0] + t1 * (p2[0] - p1[0]), p1[1] + t1 * (p2[1] - p1[1])]
        results.append((t0, compute_bearing(a, b)))
    return results


# Short route (~220 km): Wichita Falls → Oklahoma City
wf  = [-98.49, 33.91]
okc = [-97.52, 35.47]

# Long route (~7900 km): Dallas → London
dal = [-96.80, 32.78]
lon = [ -0.13, 51.51]

print("Wichita Falls → Oklahoma City (~220 km)")
print(f"  Initial bearing: {compute_bearing(wf, okc):.2f}°")
print(f"  {'Step':>6}  {'Bearing':>10}  {'Δ from initial':>16}")
wf_path = bearing_along_path(wf, okc, n=8)
initial = wf_path[0][1]
for frac, brg in wf_path:
    print(f"  {frac:>5.0%}   {brg:>9.2f}°  {brg - initial:>+15.2f}°")

print()
print("Dallas → London (~7900 km)")
print(f"  Initial bearing: {compute_bearing(dal, lon):.2f}°")
print(f"  {'Step':>6}  {'Bearing':>10}  {'Δ from initial':>16}")
dal_path = bearing_along_path(dal, lon, n=8)
initial2 = dal_path[0][1]
for frac, brg in dal_path:
    print(f"  {frac:>5.0%}   {brg:>9.2f}°  {brg - initial2:>+15.2f}°")

The short route drifts only a fraction of a degree — negligible. The long route to London drifts noticeably. By the time you are halfway across the Atlantic, the compass heading has shifted by several degrees from what it was at departure.

This is not a formula error. It is a geometric property of the sphere.

## 3. Longitude Convergence — Why the Heading Shifts

The root cause is that **lines of longitude are not parallel**. They fan out from the equator and converge at both poles. 

Imagine two longitude lines: `-97°` and `-96°`. At the equator they are about 111 km apart. At 35°N (Texas) they are about 91 km apart. At 60°N (southern Canada) they are about 56 km apart.

As you travel north along any path that is not exactly due north, you are also moving between longitude lines that are getting closer together. Your angular relationship to "due north" changes at every step — even if your physical trajectory is a smooth arc. That change in angular relationship *is* the bearing drift.

In [ ]:
# Longitude spacing at different latitudes — demonstrates convergence
KM_PER_DEG_LAT = 111.32

print(f"{'Latitude':>10}  {'km per 1° lon':>15}  {'% of equator':>14}")
print("-" * 45)
for lat in range(0, 81, 10):
    km = KM_PER_DEG_LAT * math.cos(math.radians(lat))
    pct = km / KM_PER_DEG_LAT * 100
    print(f"{lat:>9}°  {km:>14.1f} km  {pct:>13.1f}%")

## 4. Visualizing the Drift

A map makes the drift concrete. The plot below shows the Dallas → London path sampled at 12 intermediate points. At each point, a short arrow indicates the local bearing at that moment. Watch how the arrows rotate from northeast at departure toward a more northerly heading as the path arcs across the Atlantic.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

N = 12
path_points = [
    [dal[0] + i/N * (lon[0] - dal[0]), dal[1] + i/N * (lon[1] - dal[1])]
    for i in range(N + 1)
]

fig, ax = plt.subplots(figsize=(10, 4))
ax.set_title("Dallas → London: bearing at each step along the path", fontsize=11)
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")

# Path line
lons_path = [p[0] for p in path_points]
lats_path = [p[1] for p in path_points]
ax.plot(lons_path, lats_path, "-", color="#aaaaaa", linewidth=1.5, zorder=1)

# Arrow at each intermediate point showing local bearing
arrow_len = 2.5  # degrees
for i in range(N):
    p  = path_points[i]
    p2 = path_points[i + 1]
    brg = compute_bearing(p, p2)
    # Convert bearing → matplotlib angle (degrees from East, CCW)
    mpl_angle = 90 - brg
    dx = arrow_len * math.cos(math.radians(mpl_angle))
    dy = arrow_len * math.sin(math.radians(mpl_angle))
    ax.annotate(
        "", xy=(p[0] + dx * 0.4, p[1] + dy * 0.4), xytext=(p[0], p[1]),
        arrowprops=dict(arrowstyle="-|>", color="#e63946", lw=1.2),
        zorder=3,
    )
    ax.text(p[0], p[1] - 1.2, f"{brg:.0f}°", ha="center", fontsize=7, color="#555555")

# Mark endpoints
ax.plot(*dal, "o", color="#457b9d", markersize=7, zorder=4)
ax.plot(*lon, "s", color="#2a9d8f", markersize=7, zorder=4)
ax.text(dal[0], dal[1] + 0.8, "Dallas", ha="center", fontsize=9)
ax.text(lon[0], lon[1] + 0.8, "London", ha="center", fontsize=9)

ax.set_xlim(-110, 15)
ax.set_ylim(20, 65)
ax.grid(True, linestyle="--", alpha=0.4)
plt.tight_layout()
plt.show()

## 5. Flat vs. Spherical: What "Straight" Means

On a flat map, a straight line between two points is the shortest path — and it has a constant bearing the whole way. On a sphere, the shortest path (a great circle) looks curved when drawn on a flat map, and its bearing changes continuously.

The two models agree well at short distances. At intercontinental distances, they diverge enough to matter.

| Model | Path shape | Bearing | Valid at |
|-------|-----------|---------|----------|
| Flat (Euclidean) | Straight line | Constant | < ~200 km |
| Spherical (great circle) | Curved arc on map | Varies along path | Any distance |

The practical consequence: if you compute a bearing and then apply it as a fixed heading for a long journey, you will not arrive at your destination. You will curve off course because the Earth has curved under you.

In [ ]:
# Compare initial bearing vs bearing at the midpoint for routes of increasing length
base = [-98.49, 33.91]  # Wichita Falls

destinations = [
    ("Altus AFB (~90 km)",       [-99.27, 34.66]),
    ("Oklahoma City (~220 km)",  [-97.52, 35.47]),
    ("Kansas City (~560 km)",    [-94.58, 39.10]),
    ("Chicago (~1400 km)",       [-87.63, 41.88]),
    ("New York (~2400 km)",      [-74.01, 40.71]),
    ("London (~8800 km)",        [ -0.13, 51.51]),
]

print(f"{'Route':<30} {'Initial brg':>12}  {'Mid-path brg':>14}  {'Drift':>8}")
print("-" * 72)
for label, dest in destinations:
    mid = [(base[0] + dest[0]) / 2, (base[1] + dest[1]) / 2]
    initial = compute_bearing(base, dest)
    mid_brg = compute_bearing(mid, dest)
    drift   = mid_brg - initial
    print(f"{label:<30} {initial:>11.1f}°  {mid_brg:>13.1f}°  {drift:>+7.1f}°")

The drift is negligible for nearby targets and grows with distance. At intercontinental ranges the heading has rotated by several degrees before you are even halfway there.

---

## Exercise A — Drift by Direction

Bearing drift is not equal in all directions. A due-east path at mid-latitudes drifts differently than a northeast path.

Using Wichita Falls as the base point, compute initial vs. midpoint bearing for each of the four routes below. Which direction shows the most drift? Which shows the least?

```python
routes = [
    ("Due East (~900 km)",       [-87.0,  33.91]),
    ("Due North (~900 km)",      [-98.49, 41.99]),
    ("Northeast (~900 km)",      [-91.0,  40.0 ]),
    ("Southeast (~900 km)",      [-91.0,  27.0 ]),
]
```

In [1]:
base = [-98.49, 33.91]

routes = [
    ("Due East (~900 km)",   [-87.0,  33.91]),
    ("Due North (~900 km)",  [-98.49, 41.99]),
    ("Northeast (~900 km)",  [-91.0,  40.0 ]),
    ("Southeast (~900 km)",  [-91.0,  27.0 ]),
]

# your code here

## Exercise B — When Does Drift Become Significant?

Using the `bearing_along_path` function from Section 2, find the approximate distance at which bearing drift first exceeds **2°** for a northeast path from Wichita Falls.

Step east and north by varying the destination longitude/latitude and report the total distance (using `haversine_km`) alongside the max drift across the path.

In [ ]:
base = [-98.49, 33.91]

# Step northeast: increase both lon and lat by the same delta
# Try deltas from 1° to 20° in steps of 1°
print(f"{'Delta':>7}  {'Distance':>12}  {'Max drift':>12}  {'Exceeds 2°?':>12}")
print("-" * 52)

for delta in range(1, 21):
    dest = [base[0] + delta, base[1] + delta]
    dist = haversine_km(base, dest)
    path = bearing_along_path(base, dest, n=20)
    initial = path[0][1]
    max_drift = max(abs(brg - initial) for _, brg in path)
    flag = "<-- here" if max_drift >= 2.0 else ""
    print(f"{delta:>6}°   {dist:>10.0f} km  {max_drift:>10.2f}°  {flag}")

## Exercise C — Map the Bearing Drift

Using the Dallas → London route, draw a map (ipyleaflet) with:
- The path as a LineString
- A marker at each of the 8 intermediate points
- Each marker labeled with its local bearing at that step

Use `fit_bounds` to frame the map to the route extent.

In [ ]:
from ipyleaflet import Map, GeoJSON

# your code here
# hint: build path_points using bearing_along_path's intermediate coords
# hint: fit_bounds expects [[south, west], [north, east]]

---

## Check Your Understanding

A guidance system for a long-range missile is programmed with a single fixed bearing computed at launch. The target is 4,000 km away on a northeast heading.

**Question:** The missile flies a straight constant-bearing course. Will it reach the target? Explain what happens geometrically as it travels, and why the error grows with distance rather than staying constant.

```python
# your answer here
```


---

## Next

In [03 — Bearing Applications](./03-Bearing_Applications.ipynb), we put bearing to work: drawing direction lines from a launch point, filtering targets by bearing sector, and combining bearing with distance to describe what is reachable from where.